# **2. Train and Test ML algorithm in Materials - Supervised Learning**

**Load and prepare a pre-featurized dataset**

In [ ]:
# Install libraries to use matminer.
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless thinc -q
!pip install numpy==1.26.4 pandas==2.2.2 matplotlib==3.8.0 -q
!pip install opencv-python==4.10.0.84 opencv-contrib-python==4.10.0.84 -q
!pip install thinc==8.1.12 -q
!pip install pyyaml six -q
!pip install "matminer[citrine]" citrination-client pymatgen -q
!git clone https://github.com/CMDDclass/MS697-material.git

In [ ]:
from matminer.datasets.convenience_loaders import load_elastic_tensor

df = load_elastic_tensor()  # load the dataset in a pandas DataFrame object
df.columns # check columns

In [ ]:
# Removing unneeded columns from the data set
# Not all data is required for modeling.
unwanted_columns = ["volume", "nsites", "compliance_tensor", "elastic_tensor",
                    "elastic_tensor_original", "K_Voigt", "G_Voigt", "K_Reuss", "G_Reuss"]
df = df.drop(unwanted_columns, axis=1)
df.head()

In [ ]:
# Add composition-based features
# A major class of featurizers available in matminer uses the chemical composition to featurize the input data.
# Let's add some composition based features to our DataFrame.

# First step : Using the conversions Featurizers in matminer to turn a String composition (our 'formula' column from before) into a pymatgen Composition.
from matminer.featurizers.conversions import StrToComposition

df = StrToComposition().featurize_dataframe(df, "formula")
df.head() # It will be possible to confirm that a new composition column is formed.

In [ ]:
# Second step : Using one of the featurizers in matminer to add a suite of descriptors to the DataFrame.
from matminer.featurizers.composition import ElementProperty

ep_feat = ElementProperty.from_preset(preset_name="magpie")
df = ep_feat.featurize_dataframe(df, col_id="composition")  # input the "composition" column to the featurizer
df.head() # It can be seen that many composition-related features are newly created in the data.

In [ ]:
# Add more composition-based features
# There are many more Composition based featurizers apart from ElementProperty that are available in the matminer.featurizers.composition.
# Let's try the ElectronegativityDiff featurizer which requires knowing the oxidation state of the various elements in the Composition.

from matminer.featurizers.conversions import CompositionToOxidComposition
from matminer.featurizers.composition import OxidationStates

df = CompositionToOxidComposition().featurize_dataframe(df, "composition") # composition

os_feat = OxidationStates()
df = os_feat.featurize_dataframe(df, "composition_oxid") # composition_oxid // add oxidation states
df.head()

In [ ]:
# Add some structure based features
from matminer.featurizers.structure import DensityFeatures

df_feat = DensityFeatures()
df = df_feat.featurize_dataframe(df, "structure")  # input the 'structure' column to the featurizer
df.head()

**Step1: Train ML algorithm with Bulk Modulus (Kvrh) data**

In [ ]:
# y - target property(bulk modulus -> 'K_VRH'), X - remove data of string type (computer cannot accept) and initially given elastic data
y = df['K_VRH'].values
excluded = ["G_VRH", "K_VRH", "elastic_anisotropy", "formula", "material_id",
            "poisson_ratio", "structure", "composition", "composition_oxid", "space_group"]
X = df.drop(excluded, axis=1)
print("There are {} possible descriptors:\n\n{}".format(X.shape[1], X.columns.values))

Use Linear Regression model from Scikit-learn

In [ ]:
#Linear regression is a linear approach for modelling the relationship between a scalar target value and one or more explanatory variables (also known as dependent and independent variables).
#For more than one, the process is called multiple linear regression.
#This term is distinct from multivariate linear regression, where multiple correlated dependent variables are predicted, rather than a single scalar variable.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# use 'LinearRegression'
lr = LinearRegression()
lr.fit(X, y)

# get fit statistics
print('training R2 = ' + str(round(lr.score(X, y), 3)))
print('training RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y, y_pred=lr.predict(X))))

In [ ]:
# To prevent overfitting, we need to check the cross-validation score
from sklearn.model_selection import KFold, cross_val_score

# Use 'KFold'(10-fold) cross validation (90% training, 10% test), shuffle the data before splitting
crossvalidation = KFold(n_splits=10, shuffle=True, random_state=1)

# compute cross validation scores('cross_val_score') for linear regression model
scores = cross_val_score(lr, X, y, scoring='neg_mean_squared_error', cv=crossvalidation, n_jobs=1)
rmse_scores = [np.sqrt(abs(s)) for s in scores]
r2_scores = cross_val_score(lr, X, y, scoring='r2', cv=crossvalidation, n_jobs=1)

#cv: the number of fold in cross-validation, n jobs: the number of CPU in parallel computational simulation
print('Cross-validation results:')
print('Folds: %i, mean R2: %.3f' % (len(scores), np.mean(np.abs(r2_scores))))
print('Folds: %i, mean RMSE: %.3f' % (len(scores), np.mean(np.abs(rmse_scores))))

Use Random Forest model


In [ ]:
#Random Forest Regression is a supervised learning algorithm that uses ensemble learning method for regression.
#Ensemble learning method is a technique that combines predictions from multiple machine learning algorithms to make a more accurate prediction than a single model.
#the trees run in parallel with no interaction amongst them. A Random Forest operates by constructing several decision trees during training time and outputting
#the mean of the classes as the prediction of all the trees.
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=50, random_state=1)

rf.fit(X, y)
print('training R2 = ' + str(round(rf.score(X, y), 3)))
print('training RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y, y_pred=rf.predict(X))))

In [ ]:
# compute cross validation scores('cross_val_score') for random forest model
r2_scores = cross_val_score(rf, X, y, scoring='r2', cv=crossvalidation, n_jobs=-1)
scores = cross_val_score(rf, X, y, scoring='neg_mean_squared_error', cv=crossvalidation, n_jobs=-1)
rmse_scores = [np.sqrt(abs(s)) for s in scores]

print('Cross-validation results:')
print('Folds: %i, mean R2: %.3f' % (len(scores), np.mean(np.abs(r2_scores))))
print('Folds: %i, mean RMSE: %.3f' % (len(scores), np.mean(np.abs(rmse_scores))))

**Step2: Test with Random Forest model**

In [ ]:
from sklearn.model_selection import train_test_split
X['formula'] = df['formula']
# It is not used for training, but it can be necessary when you want to confirm the formula of the sample
# If we don't do this process, we can't track the formula data after tain/test split('train_test_split')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
train_formula = X_train['formula']
X_train = X_train.drop('formula', axis=1)
test_formula = X_test['formula']
X_test = X_test.drop('formula', axis=1)

# Use Random Forest model(RandomForestRegressor)
rf_reg = RandomForestRegressor(n_estimators=50, random_state=1)
rf_reg.fit(X_train, y_train)

# get fit statistics
print('training R2 = ' + str(round(rf_reg.score(X_train, y_train), 3)))
print('training RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y_train, y_pred=rf_reg.predict(X_train))))
print('test R2 = ' + str(round(rf_reg.score(X_test, y_test), 3)))
print('test RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y_test, y_pred=rf_reg.predict(X_test))))

In [ ]:
df_sample = df.loc[[0], :]
excluded = ["G_VRH", "K_VRH", "elastic_anisotropy", "formula", "material_id",
            "poisson_ratio", "structure", "composition", "composition_oxid", "space_group"]
Z = df_sample.drop(excluded, axis=1)
y_pred=rf_reg.predict(Z)

y_pred

We can optimize various parameters for machine learning algorithm by using AutoML task. In this section, we will performn AutoML through "FLAML(Fast and Lightweight AutoML".

In [ ]:
!pip install flaml -q

from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Create and train the 'AutoML' model
automl = AutoML()
automl.fit(X_train, y_train, task="regression", estimator_list=["rf"], time_budget=60)

# get fit statistics
print('training R2 = ' + str(round(automl.score(X_train, y_train), 3)))
print('training RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y_train, y_pred=automl.predict(X_train))))

[flaml.automl.logger: 09-29 05:10:25] {2466} INFO -  at 33.2s,	estimator rf's best error=0.0828,	best estimator rf's best error=0.0828
[flaml.automl.logger: 09-29 05:10:25] {2282} INFO - iteration 17, current learner rf


In [ ]:
y_pred_automl = automl.predict(Z)
print(f"Best config: {automl.best_config}")
print('test R2 = ' + str(round(automl.score(X_test, y_test), 3)))
print('test RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y_test, y_pred=automl.predict(X_test))))

y_pred_automl

**Predict a new material not contatined in the dataframe**

In [ ]:
!pip install mp_api -q

from mp_api.client import MPRester
import pandas as pd

# Replace "your_api_key_here" with your own API key from the Materials Project
mpr = MPRester("CiInKG9AGVmp2w8U1vDl9U2KI4hK9dDF")
list_of_available_fields = mpr.materials.summary.available_fields

# Search for materials with formula Fe2O3 and retrieve all available fields
docs = mpr.materials.summary.search(formula="Fe2O3", fields=list_of_available_fields)
results = [doc.dict() for doc in docs]
df_full = pd.DataFrame(results)

df2 = df_full.loc[:, ["formula_pretty", "material_id", "formation_energy_per_atom", "structure"]]
sdf = df2.sort_values(by="formation_energy_per_atom")
sdf.head()

In [ ]:
#extracting wanted material data row
df3 = df2[df2["material_id"] == "mp-19770"]
df3

In [ ]:
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.conversions import CompositionToOxidComposition
from matminer.featurizers.composition import OxidationStates
from matminer.featurizers.structure import DensityFeatures
from pymatgen.core.structure import Structure # import the Structure class from pymatgen

df3 = StrToComposition().featurize_dataframe(df3, "formula_pretty")

ep_feat = ElementProperty.from_preset(preset_name="magpie")
df3 = ep_feat.featurize_dataframe(df3, col_id="composition")  # input the "composition" column to the featurizer

df3 = CompositionToOxidComposition().featurize_dataframe(df3, "composition") # compositon -> composition_oxid // add oxidation states

os_feat = OxidationStates()
df3 = os_feat.featurize_dataframe(df3, "composition_oxid")

# Convert dictionaries in "structure" column to pymatgen Structure objects
df3["structure"] = df3["structure"].apply(lambda x: Structure.from_dict(x))

df3_feat = DensityFeatures()
df3 = df3_feat.featurize_dataframe(df3, "structure")  # input the structure column to the featurizer

In [ ]:
excluded = ["material_id", "formation_energy_per_atom", "formula_pretty", "composition", "composition_oxid", "structure"]
Z = df3.drop(excluded, axis=1)
print("There are {} possible descriptors:\n\n{}".format(Z.shape[1], Z.columns.values))

In [ ]:
Z.columns

In [ ]:
y_pred2 = rf_reg.predict(Z)
y_pred2

In [ ]:
y_pred_automl2=automl.predict(Z)
y_pred_automl2

# **3. Train and Test ML algorithm in Materials - Unsupervised Learning**

In [ ]:
!pip install matminer[citrine] -q # install matminer library
!pip install pyyaml -q
!pip install pymatgen -q

In [ ]:
# If you want to upload files directly, use the following code for a simple file upload
import pandas as pd

df = pd.read_excel('/content/MS697-material/data/modulus.xlsx')
df.head()

In [ ]:
excluded = ["material_id"]
X = df.drop(excluded, axis=1)

In [ ]:
#The agglomerative clustering is the most common type of hierarchical clustering used to group objects in clusters based on their similarity.
#It’s also known as AGNES (Agglomerative Nesting). The algorithm starts by treating each object as a singleton cluster.
#Next, pairs of clusters are successively merged until all clusters have been merged into one big cluster containing all objects.
import time as time
import matplotlib.pyplot as plt
import numpy as np


from sklearn.cluster import AgglomerativeClustering
import sklearn.datasets
#The method of merging two clusters involves combining the two clusters that result in the smallest increase in the variance within all clusters.(Ward)
print("Compute unstructured hierarchical clustering...")
st = time.time()
ward = AgglomerativeClustering(n_clusters=6, linkage="ward").fit(X)
elapsed_time = time.time() - st
group = ward.labels_
print(f"Elapsed time: {elapsed_time:.2f}s")
print(f"Number of points: {group.size}")

In [ ]:
group

In [ ]:
df['group'] = group.tolist()
df

In [ ]:
#AgglomerativeClustering --> many clusters --> connectivity constraints 필요 (only adjacent clusters can be merged together), through a connectivity matrix that defines for each sample the neighboring samples following a given structure of the data.
#For instance, in the swiss-roll example below, the connectivity constraints forbid the merging of points that are not adjacent on the swiss roll, and thus avoid forming clusters that extend across overlapping folds of the roll.
#The connectivity constraints are imposed via an connectivity matrix: a scipy sparse matrix that has elements only at the intersection of a row and a column with indices of the dataset that should be connected
from sklearn.neighbors import kneighbors_graph

connectivity = kneighbors_graph(X, n_neighbors=10, include_self=False)
print("Compute structured hierarchical clustering...")
st = time.time()
ward = AgglomerativeClustering(
    n_clusters=6, connectivity=connectivity, linkage="ward"
).fit(X)
elapsed_time = time.time() - st
group = ward.labels_
print(f"Elapsed time: {elapsed_time:.2f}s")
print(f"Number of points: {group.size}")

#Connectivity constraints and single, complete or average linkage can enhance the ‘rich getting richer’ aspect of agglomerative clustering, particularly so if they are built with sklearn.neighbors.kneighbors_graph.
#In the limit of a small number of clusters, they tend to give a few macroscopically occupied clusters and almost empty ones. (see the discussion in Agglomerative clustering with and without structure).
#Single linkage is the most brittle linkage option with regard to this issue.

In [ ]:
group

In [ ]:
df['group'] = group.tolist()

print(df)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering

X_vis = np.column_stack([np.arange(len(X)), X.values.ravel()])
x_label, y_label = "Index", X.columns[0]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, n_clusters in zip(axes.flatten(), [2, 3, 4, 5]):
    model = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
    labels = model.fit_predict(X)

    scatter = ax.scatter(X_vis[:, 0], X_vis[:, 1], c=labels, cmap="tab10", s=10)
    ax.set_title(f"n_clusters = {n_clusters}")
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)

plt.tight_layout()
plt.show()
